# VoicePD — Robust OpenSMILE + XGBoost Pipeline
### Handles real-world microphone noise (phone + laptop)
**Run cells in order. Each cell is self-contained with comments.**

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# noisereduce  — spectral noise gating
# librosa      — audio loading, VAD, resampling
# soundfile    — fallback audio reader
# opensmile    — eGeMAPS feature extraction
# xgboost      — classifier
# imblearn     — SMOTE for class imbalance handling
# ============================================================

!pip install opensmile xgboost scikit-learn pandas numpy \
             noisereduce librosa soundfile imbalanced-learn \
             joblib tqdm -q

print("All dependencies installed.")

In [ ]:
# ============================================================
# CELL 2: Imports & Global Config
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import noisereduce as nr
import opensmile
import joblib
import xgboost as xgb
import tempfile

from tqdm import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

# ── Global audio config ──────────────────────────────────────
TARGET_SR       = 16000   # HuBERT & eGeMAPS both expect 16kHz
MIN_DURATION_S  = 1.5     # Reject clips shorter than this (seconds)
MAX_DURATION_S  = 12.0    # Trim clips longer than this
TOP_DB          = 30      # Silence trim threshold (dB below peak)
NOISE_REDUCE    = True    # Toggle spectral noise reduction
RANDOM_STATE    = 42

print("Imports OK | Target SR:", TARGET_SR, "Hz")

In [ ]:
# ============================================================
# CELL 3: Audio Preprocessing Pipeline
#
# Steps applied to EVERY audio file (training + inference):
#   1. Load with librosa (handles mp3/wav/ogg/m4a)
#   2. Convert to mono
#   3. Resample to TARGET_SR (16kHz)
#   4. Normalize amplitude to [-1, 1]
#   5. Trim leading/trailing silence (VAD-lite)
#   6. Spectral noise reduction
#   7. Duration gate — reject too short, trim too long
# ============================================================

def load_and_preprocess(file_path, augment=False):
    """
    Load an audio file and apply the full preprocessing chain.

    Parameters
    ----------
    file_path : str
        Path to any audio file (wav, mp3, ogg, m4a, flac).
    augment : bool
        If True, apply random noise + pitch augmentation.
        Used during training only — never during inference.

    Returns
    -------
    np.ndarray or None
        Preprocessed audio signal at TARGET_SR, or None if rejected.
    """
    try:
        # ── 1. Load (librosa handles format detection automatically) ──
        audio, sr = librosa.load(file_path, sr=None, mono=True)

        # ── 2. Resample to 16kHz if needed ───────────────────────────
        if sr != TARGET_SR:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)

        # ── 3. Amplitude normalisation (peak normalise to ±1) ────────
        peak = np.max(np.abs(audio))
        if peak > 0:
            audio = audio / peak

        # ── 4. Silence trim (removes dead air at start/end) ──────────
        #    top_db=30 means anything >30dB below peak is silence
        audio, _ = librosa.effects.trim(audio, top_db=TOP_DB)

        # ── 5. Duration gate ─────────────────────────────────────────
        duration = len(audio) / TARGET_SR
        if duration < MIN_DURATION_S:
            # Too short after trimming — likely silence or corrupt
            return None
        if duration > MAX_DURATION_S:
            # Trim to max duration from the centre (keeps the steadiest part)
            max_samples = int(MAX_DURATION_S * TARGET_SR)
            start = (len(audio) - max_samples) // 2
            audio = audio[start : start + max_samples]

        # ── 6. Spectral noise reduction ───────────────────────────────
        #    Estimates noise profile from first 0.3s (assumed to be
        #    background noise before speech begins), then suppresses it.
        if NOISE_REDUCE:
            noise_sample_len = int(0.3 * TARGET_SR)
            noise_profile = audio[:noise_sample_len]
            audio = nr.reduce_noise(
                y=audio,
                sr=TARGET_SR,
                y_noise=noise_profile,
                prop_decrease=0.75,   # Reduce noise by 75% (not 100% — avoids musical noise)
                stationary=False      # Non-stationary handles variable background noise
            )

        # ── 7. Training augmentation (DISABLED during inference) ──────
        if augment:
            audio = _augment_audio(audio, TARGET_SR)

        return audio

    except Exception as e:
        print(f"  [SKIP] {os.path.basename(file_path)}: {e}")
        return None


def _augment_audio(audio, sr):
    """
    Apply random augmentations to simulate real-world recording conditions.
    Only used during TRAINING — never during inference.

    Augmentations (each applied randomly):
      - Gaussian noise addition (simulates microphone hiss)
      - Slight pitch shift ±1.5 semitones (natural voice variation)
      - Slight speed perturbation ±5% (recording rate variation)
    """
    # Random Gaussian noise (SNR ~25-40dB — realistic indoor mic noise)
    if np.random.rand() < 0.5:
        noise_level = np.random.uniform(0.003, 0.012)
        audio = audio + noise_level * np.random.randn(len(audio))

    # Pitch shift (preserves duration — important for eGeMAPS features)
    if np.random.rand() < 0.4:
        n_steps = np.random.uniform(-1.5, 1.5)
        audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)

    # Speed perturbation (slight tempo change without pitch change)
    if np.random.rand() < 0.3:
        rate = np.random.uniform(0.95, 1.05)
        audio = librosa.effects.time_stretch(audio, rate=rate)
        # Re-trim to max duration after stretch
        max_samples = int(MAX_DURATION_S * sr)
        audio = audio[:max_samples]

    # Re-normalise after augmentation
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak

    return audio


print("Preprocessing pipeline defined.")

In [ ]:
# ============================================================
# CELL 4: OpenSMILE Feature Extractor
#
# eGeMAPSv02 extracts 88 functionals covering:
#   - F0 (pitch) stats — reduced/monotone pitch is a PD marker
#   - Energy/loudness — hypophonia (reduced volume) in PD
#   - Spectral features — shimmer, jitter, HNR
#   - MFCC-related — vocal tract shape changes
#
# We write the preprocessed audio to a temp WAV file because
# openSMILE's process_file() needs a file path, not a numpy array.
# ============================================================

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals
)

FEATURE_NAMES = smile.feature_names
print(f"eGeMAPSv02 features: {len(FEATURE_NAMES)} functionals")


def extract_features(audio_array, sr=TARGET_SR):
    """
    Extract eGeMAPSv02 features from a preprocessed numpy audio array.

    Writes to a temp file, extracts, then deletes the temp file.
    This is the correct way to use openSMILE with preprocessed audio.

    Returns
    -------
    np.ndarray of shape (88,) or None on failure.
    """
    try:
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
            tmp_path = tmp.name
        sf.write(tmp_path, audio_array, sr, subtype='PCM_16')
        df = smile.process_file(tmp_path)
        features = df.values.flatten().astype(np.float32)
        os.unlink(tmp_path)  # Clean up temp file
        return features
    except Exception as e:
        try:
            os.unlink(tmp_path)
        except:
            pass
        print(f"  [FEAT ERROR]: {e}")
        return None


def extract_features_from_file(file_path, augment=False):
    """
    Full pipeline: load → preprocess → extract features.
    Single entry point used for both training and inference.
    """
    audio = load_and_preprocess(file_path, augment=augment)
    if audio is None:
        return None
    return extract_features(audio)


print("Feature extractor ready.")

In [ ]:
# ============================================================
# CELL 5: Dataset Loader
#
# Loads all WAV files from your folder structure.
# Set AUGMENT_TRAINING=True to generate augmented copies
# of the minority class (Parkinson's) — helps with imbalance.
#
# CHANGE: Update DATA_DIR to your actual path.
# ============================================================

DATA_DIR         = "/kaggle/input/datasets/sakshamyadav15/parkinson-data/data"
AUGMENT_TRAINING = True   # Set False to disable augmentation during load
AUG_COPIES       = 1      # How many augmented copies per PD file (1 = doubles PD data)

X, y = [], []
skipped = 0


def process_folder(folder_path, label, augment=False, aug_copies=1):
    global skipped
    files = []
    for root, _, fnames in os.walk(folder_path):
        for f in fnames:
            if f.lower().endswith(('.wav', '.mp3', '.ogg', '.flac')):
                files.append(os.path.join(root, f))

    label_name = 'Healthy' if label == 0 else 'Parkinsons'
    print(f"  Processing {len(files)} files → [{label_name}] from {os.path.basename(folder_path)}")

    for path in tqdm(files, leave=False):
        # Original (no augmentation)
        feats = extract_features_from_file(path, augment=False)
        if feats is not None:
            X.append(feats)
            y.append(label)
        else:
            skipped += 1

        # Augmented copies (only if requested)
        if augment and feats is not None:
            for _ in range(aug_copies):
                feats_aug = extract_features_from_file(path, augment=True)
                if feats_aug is not None:
                    X.append(feats_aug)
                    y.append(label)


print("Loading HEALTHY samples...")
process_folder(os.path.join(DATA_DIR, "Italian/15_Young_Healthy_Control"), 0)
process_folder(os.path.join(DATA_DIR, "Italian/22_Elderly_Healthy_Control"), 0)
process_folder(os.path.join(DATA_DIR, "figdata/HC_AH"), 0)
process_folder(os.path.join(DATA_DIR, "figdata/HC_External"), 0)

print("\nLoading PARKINSON'S samples...")
# Augment PD class to help balance the dataset
process_folder(os.path.join(DATA_DIR, "Italian/28_People_with_Parkinsons"), 1,
               augment=AUGMENT_TRAINING, aug_copies=AUG_COPIES)
process_folder(os.path.join(DATA_DIR, "figdata/PD_AH"), 1,
               augment=AUGMENT_TRAINING, aug_copies=AUG_COPIES)
process_folder(os.path.join(DATA_DIR, "gitdata/Parkinson"), 1,
               augment=AUGMENT_TRAINING, aug_copies=AUG_COPIES)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print(f"\n{'='*50}")
print(f"Total samples  : {len(y)}")
print(f"Healthy        : {np.sum(y == 0)}")
print(f"Parkinsons     : {np.sum(y == 1)}")
print(f"Skipped files  : {skipped}")
print(f"Feature dim    : {X.shape[1]}")
print(f"NaN in features: {np.isnan(X).sum()}")

# Replace any NaN/inf with 0 (robustness for edge-case recordings)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
print("NaN/Inf cleaned. Dataset ready.")

In [ ]:
# ============================================================
# CELL 6: Split, Scale, SMOTE
#
# - Stratified split preserves class ratio in train/test
# - StandardScaler: eGeMAPS features have very different ranges,
#   scaling is CRITICAL for XGBoost to work well
# - SMOTE: generates synthetic minority class samples in
#   feature space (applied ONLY to training set — never test)
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

# Scale BEFORE SMOTE (SMOTE interpolates in scaled space)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled  = scaler.transform(X_test)         # apply same transform to test

print(f"Before SMOTE → Train: {X_train_scaled.shape}, "
      f"Healthy: {np.sum(y_train==0)}, PD: {np.sum(y_train==1)}")

# SMOTE to handle class imbalance
# k_neighbors=3 is safer when PD class is small
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print(f"After  SMOTE → Train: {X_train_bal.shape}, "
      f"Healthy: {np.sum(y_train_bal==0)}, PD: {np.sum(y_train_bal==1)}")
print(f"Test set       → {X_test_scaled.shape} (untouched, no SMOTE on test)")

In [ ]:
# ============================================================
# CELL 7: Train XGBoost Model
#
# Key parameters explained:
#   n_estimators    : 500 trees — more than your original 400
#   max_depth       : 5 — prevents overfitting on small dataset
#   learning_rate   : 0.03 — slower learning = better generalisation
#   subsample       : 0.85 — row subsampling per tree
#   colsample_bytree: 0.75 — feature subsampling per tree
#   min_child_weight: 3 — min samples in leaf (regularisation)
#   gamma           : 0.1 — min loss reduction to split
#   reg_alpha/lambda: L1/L2 regularisation
#   scale_pos_weight: adjusts for residual imbalance after SMOTE
#
# Early stopping: stops training if val AUC doesn't improve
# for 40 rounds — prevents overfitting automatically.
# ============================================================

# Residual class ratio after SMOTE (should be ~1.0 if balanced)
pos_weight = np.sum(y_train_bal == 0) / np.sum(y_train_bal == 1)

model = xgb.XGBClassifier(
    n_estimators        = 500,
    max_depth           = 5,
    learning_rate       = 0.03,
    subsample           = 0.85,
    colsample_bytree    = 0.75,
    min_child_weight    = 3,
    gamma               = 0.1,
    reg_alpha           = 0.1,    # L1 regularisation
    reg_lambda          = 1.5,    # L2 regularisation
    scale_pos_weight    = pos_weight,
    eval_metric         = 'auc',
    early_stopping_rounds = 40,
    random_state        = RANDOM_STATE,
    n_jobs              = -1
)

# Use 15% of training data as validation for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_bal, y_train_bal,
    test_size=0.15,
    stratify=y_train_bal,
    random_state=RANDOM_STATE
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=50   # Print every 50 rounds
)

print(f"\nTraining complete. Best iteration: {model.best_iteration}")

In [ ]:
# ============================================================
# CELL 8: Evaluation
#
# We find the OPTIMAL decision threshold using the ROC curve
# instead of hardcoding 0.5.
#
# For a screening tool, we want HIGH sensitivity (catch as
# many PD cases as possible) — so we bias the threshold
# slightly toward sensitivity using Youden's J statistic.
# ============================================================

y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Find optimal threshold via Youden's J = sensitivity + specificity - 1
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
j_scores = tpr - fpr
best_thresh = thresholds[np.argmax(j_scores)]

print(f"Optimal decision threshold (Youden's J): {best_thresh:.4f}")
print(f"(Default 0.5 is replaced by this — better sensitivity/specificity balance)\n")

y_pred = (y_prob >= best_thresh).astype(int)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
cm  = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print(f"AUC-ROC       : {auc:.4f}")
print(f"Accuracy      : {acc:.4f}")
print(f"Sensitivity   : {sensitivity:.4f}   ← catching PD cases")
print(f"Specificity   : {specificity:.4f}   ← avoiding false alarms")
print(f"\nConfusion Matrix:")
print(f"  TN={tn}  FP={fp}")
print(f"  FN={fn}  TP={tp}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Healthy', 'Parkinsons']))

In [ ]:
# ============================================================
# CELL 9: Cross-Validation (Sanity Check)
#
# 5-fold stratified CV on the FULL dataset (before SMOTE)
# tells you how stable the model is — if CV AUC is much
# lower than test AUC, you have an overfitting problem.
# Target: CV AUC within ~0.02 of test AUC.
# ============================================================

print("Running 5-fold cross-validation (this takes a few minutes)...")

X_scaled_full = scaler.transform(X)   # Scale full dataset
X_scaled_full = np.nan_to_num(X_scaled_full)

cv_model = xgb.XGBClassifier(
    n_estimators     = model.best_iteration,   # Use best iteration from training
    max_depth        = 5,
    learning_rate    = 0.03,
    subsample        = 0.85,
    colsample_bytree = 0.75,
    min_child_weight = 3,
    gamma            = 0.1,
    reg_alpha        = 0.1,
    reg_lambda       = 1.5,
    eval_metric      = 'auc',
    random_state     = RANDOM_STATE,
    n_jobs           = -1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(cv_model, X_scaled_full, y, cv=cv,
                             scoring='roc_auc', n_jobs=-1)

print(f"\nCV AUC scores : {cv_scores.round(4)}")
print(f"Mean CV AUC   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Test AUC      : {auc:.4f}")
gap = abs(auc - cv_scores.mean())
if gap < 0.03:
    print(f"Gap: {gap:.4f} → Model generalises well ✓")
elif gap < 0.06:
    print(f"Gap: {gap:.4f} → Mild overfitting — consider more regularisation")
else:
    print(f"Gap: {gap:.4f} → Significant overfitting — increase reg_alpha/lambda")

In [ ]:
# ============================================================
# CELL 10: Feature Importance
#
# Shows which eGeMAPS features matter most.
# Top features should be clinically meaningful:
#   - F0 (pitch) variability — PD = monotone
#   - Shimmer/Jitter — PD = irregular vibration
#   - HNR — PD = breathier voice
#   - Loudness — PD = hypophonia
# ============================================================

importances = model.feature_importances_
feat_df = pd.DataFrame({
    'feature'   : FEATURE_NAMES,
    'importance': importances
}).sort_values('importance', ascending=False)

print("Top 15 most important features:")
print(feat_df.head(15).to_string(index=False))

# Save feature importance CSV for paper/writeup
feat_df.to_csv('feature_importance.csv', index=False)
print("\nSaved to feature_importance.csv")

In [ ]:
# ============================================================
# CELL 11: Save Model Artifacts
#
# Saves everything needed for deployment:
#   opxg_model.json   — XGBoost model weights
#   scaler.pkl        — StandardScaler (MUST use same scaler at inference)
#   threshold.txt     — optimal decision threshold found in Cell 8
#
# IMPORTANT: All 3 files must travel together to deployment.
# ============================================================

model.save_model('opxg_model.json')
joblib.dump(scaler, 'scaler.pkl')

with open('threshold.txt', 'w') as f:
    f.write(str(best_thresh))

# Save feature names so inference can verify feature order
pd.Series(FEATURE_NAMES).to_csv('feature_names.csv', index=False, header=False)

print("Saved:")
print("  opxg_model.json     — model weights")
print("  scaler.pkl          — feature scaler")
print("  threshold.txt       — optimal decision threshold")
print("  feature_names.csv   — feature order reference")

In [ ]:
# ============================================================
# CELL 12: Inference Function (Production-Ready)
#
# This is what your app calls. It:
#   1. Loads model + scaler + threshold from saved files
#   2. Applies the EXACT same preprocessing as training
#   3. Returns: label, probability, confidence band, and
#      a user-friendly message (for the app UI)
#
# Confidence bands:
#   HIGH   : prob > 0.75 or < 0.25  — model is certain
#   MEDIUM : prob 0.55-0.75 or 0.25-0.45 — moderate confidence  
#   LOW    : prob 0.45-0.55 — near the threshold, uncertain
#            → LOW confidence always recommends clinical visit
# ============================================================

class VoicePDPredictor:
    """
    Production inference wrapper for VoicePD.
    Handles all preprocessing internally — just pass a file path.
    """

    def __init__(self, model_path='opxg_model.json',
                 scaler_path='scaler.pkl',
                 threshold_path='threshold.txt'):

        self.model = xgb.XGBClassifier()
        self.model.load_model(model_path)

        self.scaler = joblib.load(scaler_path)

        with open(threshold_path) as f:
            self.threshold = float(f.read().strip())

        print(f"VoicePDPredictor loaded | threshold={self.threshold:.4f}")


    def predict(self, file_path):
        """
        Run full inference on a single audio file.

        Returns
        -------
        dict with keys:
            label         : 'Healthy' or 'Parkinsons'
            probability   : float 0-1 (PD probability)
            confidence    : 'HIGH' | 'MEDIUM' | 'LOW'
            message       : User-friendly string for app display
            error         : None or error string if something went wrong
        """
        # Step 1: Preprocess (same pipeline as training)
        audio = load_and_preprocess(file_path, augment=False)
        if audio is None:
            return {
                'label': None, 'probability': None,
                'confidence': None,
                'message': 'Recording too short or silent. Please try again in a quiet environment.',
                'error': 'preprocessing_failed'
            }

        # Step 2: Extract features
        features = extract_features(audio)
        if features is None:
            return {
                'label': None, 'probability': None,
                'confidence': None,
                'message': 'Could not analyse audio. Please check microphone and try again.',
                'error': 'feature_extraction_failed'
            }

        # Step 3: Scale + clean
        features = np.nan_to_num(features, nan=0.0).reshape(1, -1)
        features_scaled = self.scaler.transform(features)

        # Step 4: Predict
        prob = float(self.model.predict_proba(features_scaled)[0][1])
        label = 'Parkinsons' if prob >= self.threshold else 'Healthy'

        # Step 5: Confidence band
        dist_from_threshold = abs(prob - self.threshold)
        if dist_from_threshold > 0.25:
            confidence = 'HIGH'
        elif dist_from_threshold > 0.10:
            confidence = 'MEDIUM'
        else:
            confidence = 'LOW'   # Near the threshold — uncertain

        # Step 6: User-facing message (for app UI)
        message = self._get_message(label, confidence, prob)

        return {
            'label'      : label,
            'probability': round(prob, 4),
            'confidence' : confidence,
            'message'    : message,
            'error'      : None
        }


    def _get_message(self, label, confidence, prob):
        """Returns plain-language message suitable for non-clinical users."""
        if confidence == 'LOW':
            return ("Your voice patterns are near our detection boundary. "
                    "We recommend speaking with a doctor for a proper evaluation.")
        if label == 'Parkinsons' and confidence == 'HIGH':
            return ("Your voice shows patterns that may be associated with Parkinson's Disease. "
                    "This is not a diagnosis — please consult a neurologist as soon as possible.")
        if label == 'Parkinsons' and confidence == 'MEDIUM':
            return ("Some vocal patterns associated with Parkinson's Disease were detected. "
                    "We recommend a check-up with your doctor.")
        if label == 'Healthy' and confidence == 'HIGH':
            return ("Your voice patterns appear typical. "
                    "If you have concerns, please still consult a doctor.")
        # Healthy MEDIUM
        return ("No strong indicators detected, but some patterns warrant attention. "
                "Consider speaking with a doctor if you have symptoms.")


# ── Quick test ────────────────────────────────────────────────
predictor = VoicePDPredictor()

# Uncomment to test on a real file:
# result = predictor.predict('path/to/test.wav')
# print(result)

print("\nVoicePDPredictor ready for deployment.")
print("Call: predictor.predict('your_audio_file.wav')")